In [ ]:
import jax

jax.config.update('jax_platform_name', 'cpu')
jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt

from gould_2026.datasets import Odoherty21Dataset, Zong22Dataset
import numpy as np
from gould_2026.save_to_cache import save_to_cache
from optimization_comparison import simulations_to_l_df
import scipy
import seaborn as sns
from gould_2026.plotting import Palette, paper_plot_context, EM, LINEWIDTH


from gould_2026.prediction.vjf import VJF
from gould_2026.prediction.bubblewrap import Bubblewrap
from gould_2026.prediction.kalman_filter import StreamingKalmanFilter
from gould_2026.stim_designer import OptimizationMethod
from gould_2026.sim_stim import StimDirectionType, StimResponseType, SimStimConfig, run_simulations

from scipy.spatial.distance import pdist, squareform
from gould_2026.utils import angle_between
from pathlib import Path



In [ ]:
rng = np.random.default_rng(0)

In [ ]:
n_runs = 6
dataset = 'odoherty21'
type_of_dim_red = 'prosvd'
type_of_autoreg = 'kf'
optimization_method = OptimizationMethod.LBFGS_UNCONSTRAINED
prosvd_k = 2
output: Path | None = None
output2_decreasing_error: Path | None = None

In [ ]:
def _f(n_runs=n_runs, args_dataset=dataset, args_type_of_dim_red=type_of_dim_red, args_type_of_autoreg=type_of_autoreg, prosvd_k=prosvd_k, optimization_method=optimization_method):
    if args_dataset == 'odoherty21':
        data = Odoherty21Dataset().neural_data
    elif args_dataset == 'zong22':
        data = Zong22Dataset().neural_data
    else:
        raise ValueError()

    if args_type_of_autoreg == 'kf':
        autoreg = StreamingKalmanFilter
    elif args_type_of_autoreg == 'bw':
        autoreg = Bubblewrap
    elif args_type_of_autoreg == 'vjf':
        autoreg = VJF
    else:
        raise ValueError()

    common = SimStimConfig(
        isi_generator=[2.],
        exit_time=np.inf,
        prosvd_k=prosvd_k,
        optimization_method=optimization_method,
        stim_direction_type=StimDirectionType.RANDOM,
        v_design_use_full_u_s_map=False,
        last_dim_red=args_type_of_dim_red,
        autoreg=autoreg,
        show_tqdm=True,
        steady_state_prosvd=True,
        init_optimization_method=OptimizationMethod.RANDOM_DENSE_GAUSSIAN
    )
    to_run = {
        'open id': common.update(u_to_s_model_type='identity', true_S=StimResponseType.IDENTITY),
        'closed id': common.update(u_to_s_model_type='kernel_regressed', true_S=StimResponseType.IDENTITY),
        'open flip': common.update(u_to_s_model_type='identity', true_S=StimResponseType.FLIP),
        'closed flip': common.update(u_to_s_model_type='kernel_regressed', true_S=StimResponseType.FLIP),
    }

    simulations = run_simulations(data=data, rng=rng, to_run=to_run, n_runs=n_runs, show_tqdm=False)
    l_df = simulations_to_l_df(simulations)
    return l_df


In [ ]:
f = save_to_cache('optim_open_vs_closed', location='/mnt/data/gould_2026_cache/')(_f)

l_df = _f()

In [ ]:

l_df[['open_closed', 'true_s']] = l_df['sr_key'].str.split(' ', expand=True)

l_df['angle(s_obs,v)'] = l_df.l.apply(lambda l: angle_between(l['observed_s_hat'], l['v']))
l_df['angle(s,v)'] = l_df.l.apply(lambda l: angle_between(l['s'], l['v']))

l_df['v'] = l_df.l.apply(lambda x: x['v'])
l_df['u'] = l_df.l.apply(lambda x: x['u'])
l_df['high_d_v'] = l_df.apply(lambda x: x['l']['equiv_proj_mat'] @ x['v'], axis=1)

l_df['theta'] = l_df['l'].apply(lambda l: angle_between(l['v'], l['observed_s_hat'], radians=False))
l_df['ones_theta'] = l_df['l'].apply(lambda l: angle_between(l['u'], l['u']*0+1, radians=False))
l_df['r'] = l_df['l'].apply(lambda l: np.linalg.norm(l['observed_s_hat']))
l_df['t'] = l_df['l'].apply(lambda l: l['time_of_stim'])


l_df['Q'] = l_df.l.apply(lambda x: x['equiv_proj_mat'])


In [ ]:
%matplotlib inline
with paper_plot_context():
    fig, axs = plt.subplots(figsize=(4, 4), nrows=2, ncols=2, sharex=True, sharey=True, squeeze=False, layout='constrained')

    min_norm = 5
    max_angle = 25

    for k, ax in zip(l_df.sr_key.unique(), axs.flatten()):
        sub_df = l_df[l_df.sr_key == k]
        sub_df = sub_df[sub_df.t > sub_df.t.median()]
        ax.scatter(sub_df['theta'], sub_df['r'], LINEWIDTH*1.5, label=k, color='k')
        patch = plt.Rectangle(xy=(0,min_norm), width=max_angle, height=200, color='r', alpha=.1)
        ax.add_patch(patch)
        ax.set_xlim(xmin=0, xmax=180)
        ax.text(0.99, 0.97, k + f"\n {((sub_df.theta < max_angle) & (sub_df.r > min_norm)).sum()} / {len(sub_df)}", transform=ax.transAxes, ha='right', va='top', fontsize=EM)

        ax.set_ylim(0,90)
        ax.axvline(90, linestyle='--', color='gray', alpha=0.5)

    for ax in axs[-1,:]:
        ax.set_xlabel('Angle between v and s (degrees)')

    for ax in axs[:,0]:
        ax.set_ylabel('Norm of s')

    if output is not None:
        fig.savefig(output, bbox_inches="tight", transparent=True)



In [ ]:
pivot = l_df.pivot(index=['sr_i', 'l_i'], columns=['sr_key'], values=['theta', 'r']).dropna()

In [ ]:
def median_and_radius(x):
    median = np.quantile(x, .5)
    r = np.quantile(np.abs(x - median), .9)
    print(f'median = {median}, r = {r}')
median_and_radius(pivot[('theta', 'closed flip')])

In [ ]:

a = pivot[('theta', 'open flip')]
b = pivot[('theta', 'closed flip')]
test_result = scipy.stats.wilcoxon(a, b)
test_result

In [ ]:

a = pivot[('theta', 'closed id')]
b = pivot[('theta', 'closed flip')]
test_result = scipy.stats.wilcoxon(a, b)
test_result

In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))
sub_df = l_df[l_df['sr_key'].isin(['open flip', 'closed flip'])]
sns.stripplot(sub_df, x='sr_key', y='theta', ax=ax, s=1, color='gray', alpha=.5)
sns.violinplot(sub_df, x='sr_key', y='theta', ax=ax, color='C0')
ax.set_title(f'p={test_result.pvalue:.3g}\t$\\Delta = {a.median() - b.median():.3g}^\\circ$')

In [ ]:
fig, ax = plt.subplots()

def pdist_plot_for_column(l_df, c, ax, method=angle_between, show_sr_boundaries=False):
    sub_df = l_df.sort_values(['sr_key', 'sr_i', 'l_i'])
    to_compare = np.squeeze(np.stack(sub_df[c]))
    distance_matrix = squareform(pdist(to_compare, method))
    matshow = ax.matshow(distance_matrix)
    ax.set_title(f'{c} vs {c}')

    block_sizes = sub_df.groupby('sr_key', sort=False).size()
    block_centers = block_sizes.cumsum() - (block_sizes / 2) - 0.5
    ax.set_xticks(block_centers.to_numpy())
    ax.set_xticklabels(block_sizes.index.to_list(), rotation=45, ha='right')

    plt.colorbar(matshow, ax=ax)

    if show_sr_boundaries:
        sr_i_values = sub_df['sr_i'].to_numpy()
        sr_i_change_boundaries = np.flatnonzero(sr_i_values[1:] != sr_i_values[:-1]) + 1
        for boundary in sr_i_change_boundaries:
            line_pos = boundary - 0.5
            ax.axvline(line_pos, color='white', linewidth=0.5)
            ax.axhline(line_pos, color='white', linewidth=0.5)
    return distance_matrix

pdist_plot_for_column(l_df.sample(100, random_state=rng), 'v', ax)


In [ ]:

fig, ax = plt.subplots()

distance_matrix = pdist_plot_for_column(l_df[(l_df['sr_key'] == 'closed id') & (l_df['r'] > 5)], 'u', ax)


In [ ]:
row_indices, column_indices = np.triu_indices_from(distance_matrix, k=1)
pairwise = distance_matrix[row_indices, column_indices]
plt.hist(pairwise, bins=np.linspace(0, 100, 100));
(pairwise < 1).mean()

In [ ]:
fig, axs = plt.subplots(squeeze=False, nrows=1, ncols=1, constrained_layout=True, figsize=(6, 6))
pdist_plot_for_column(l_df.sample(1000), 'high_d_v', axs[0,0])

In [ ]:

fig7, axs6 = plt.subplots(squeeze=False, nrows=1, ncols=1, constrained_layout=True, figsize=(6, 6))
pdist_plot_for_column(l_df.sample(1000), 'v', axs6[0,0])

In [ ]:

fig7, axs7 = plt.subplots(squeeze=False, nrows=1, ncols=1, constrained_layout=True, figsize=(6,6))
sns.scatterplot(data=l_df.sample(1000), x='theta', y='ones_theta', hue='sr_key', ax=axs7[0,0])

In [ ]:
with paper_plot_context():
    fig, ax = plt.subplots(figsize=(3, 3))

    for i, sr_key in enumerate(l_df.sr_key.unique()):
        sub_df = l_df[l_df['sr_key'] == sr_key]
        ax.scatter(sub_df['t'], sub_df['theta'], color=f'C{i}', marker='.', s=2, alpha=.3)

    x = np.array([l_df.t.min(), l_df.t.max()])
    for i, sr_key in enumerate(l_df.sr_key.unique()):
        sub_df = l_df[l_df['sr_key'] == sr_key]
        reg = scipy.stats.linregress(sub_df['t'], sub_df['theta'])
        ax.plot(x, x*reg.slope + reg.intercept, color=f'C{i}', linewidth=2*LINEWIDTH)

    ax.set_xlabel('time')
    ax.set_ylabel(r'error, $\theta$ (degrees)')

    if output2_decreasing_error is not None:
        fig.savefig(output2_decreasing_error)



In [ ]:
reg

In [ ]:
sns.scatterplot(data=l_df[l_df['open_closed'] == 'closed'], x='angle(s,v)', y='angle(s_obs,v)', hue='sr_key', markers='.' )